<a href="https://colab.research.google.com/github/IvanSSantana/ai-academic-search/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SETUP DO OLLAMA (IA LOCAL)

In [2]:
import subprocess

subprocess.run(["apt-get", "install", "zstd"])

CompletedProcess(args=['apt-get', 'install', 'zstd'], returncode=0)

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Após a instalação do Ollama acima, digite no terminal 'ollama serve' e rode os dois comandos abaixo, entretanto espere a instalação com 'ollama pull' antes de rodar 'ollama run', para isto observe no terminal os packages sendo instalados

In [4]:
import os
import subprocess

def start_ollama_server():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    subprocess.Popen(["ollama", "pull", "llama3.1"])

start_ollama_server()

In [5]:
import os
import subprocess

def start_ollama_server():
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'
    subprocess.Popen(["ollama", "run", "llama3.1"])

start_ollama_server()

# APP

In [1]:
!pip install agno ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.3 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=c4b706618cad17287129818fb4ecf345357158e0e5d02aecb17eabca5ead1810
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


In [20]:
import time
import requests
import xml.etree.ElementTree as ET
from agno.models.ollama import Ollama
from agno.agent import Agent
from agno.tools import tool

# ---------------------------------------------------------------------------
# Utilitário para a API do PubMed (E-utilities) – permanece igual
# ---------------------------------------------------------------------------

NCBI_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
NCBI_EMAIL = "seu_email@example.com"  # ⚠️ Substitua pelo seu e‑mail real

def _fazer_consulta_pubmed(query: str, max_results: int = 5) -> list[dict]:
    """
    Executa ESearch + EFetch na PubMed e devolve uma lista de dicionários
    com título, autores, data, resumo e PMID de cada artigo.
    """
    params_esearch = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "usehistory": "y",
        "retmode": "json",
        "email": NCBI_EMAIL,
        "tool": "AgnoAgent"
    }
    try:
        resp = requests.get(f"{NCBI_BASE}/esearch.fcgi", params=params_esearch)
        resp.raise_for_status()
        data_esearch = resp.json()
        id_list = data_esearch.get("esearchresult", {}).get("idlist", [])
        if not id_list:
            return []

        params_efetch = {
            "db": "pubmed",
            "id": ",".join(id_list),
            "rettype": "abstract",
            "retmode": "xml",
            "email": NCBI_EMAIL,
            "tool": "AgnoAgent"
        }
        time.sleep(0.5)
        resp_efetch = requests.get(f"{NCBI_BASE}/efetch.fcgi", params=params_efetch)
        resp_efetch.raise_for_status()

        root = ET.fromstring(resp_efetch.content)
        artigos = []
        for article in root.findall(".//PubmedArticle"):
            pmid = article.findtext(".//PMID")
            title = article.findtext(".//ArticleTitle") or "Título não disponível"
            abstract = article.findtext(".//AbstractText") or "Resumo não disponível"
            pub_date = ""
            pub_day = article.findtext(".//PubMedPubDate/Day")
            pub_month = article.findtext(".//PubMedPubDate/Month")
            pub_year = article.findtext(".//PubMedPubDate/Year")
            if pub_year:
                pub_date = f"{pub_day or '01'}/{pub_month or '01'}/{pub_year}"
            autores = []
            for auth in article.findall(".//Author"):
                last = auth.findtext("./LastName") or ""
                fore = auth.findtext("./ForeName") or ""
                if last:
                    autores.append(f"{last} {fore}".strip())
            autores_str = ", ".join(autores[:5]) if autores else "Autores não disponíveis"

            artigos.append({
                "pmid": pmid,
                "titulo": title,
                "autores": autores_str,
                "data": pub_date,
                "resumo": abstract[:500] + ("..." if len(abstract) > 500 else ""),
                "link": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
            })
        return artigos

    except Exception as e:
        print(f"WARN: Erro na consulta ao PubMed: {e}")
        return []


# ---------------------------------------------------------------------------
# Extrator de palavras‑chave (modificado para aceitar contexto de exclusão)
# ---------------------------------------------------------------------------

def extrair_palavras_chave(prompt: str, palavras_excluidas: list[str] | None = None) -> str:
    """
    Gera ATÉ 2 palavras‑chave em inglês, garantindo que sejam diferentes
    das palavras fornecidas em `palavras_excluidas`.
    """
    if palavras_excluidas is None:
        palavras_excluidas = []

    instrucao_exclusao = ""
    if palavras_excluidas:
        instrucao_exclusao = (
            "As seguintes palavras NÃO podem ser usadas: " + ", ".join(palavras_excluidas) + ".\n"
            "Gere termos completamente diferentes desses.\n"
        )

    agente_palavras = Agent(
        model=Ollama(id="llama3.1", options={"temperature": 0.11}),
        instructions=(
            "Você é um especialista em farmacologia esportiva e análise crítica de discursos "
            "sobre esteroides anabólicos androgênicos (AAS), aminoácidos essenciais (EAA) e suplementos.\n"
            "Dado um tópico ou trecho de discurso, extraia ATÉ 2 palavras‑chave em inglês, "
            "separadas por espaços, que capturem os conceitos biomédicos mais relevantes para "
            "uma busca na base de dados PubMed.\n"
            + instrucao_exclusao +
            "Foque em termos como: mecanismos de ação, efeitos adversos, ensaios clínicos, "
            "falácias lógicas, riscos cardiovasculares/hormonais, evidência anedótica, etc.\n"
            "Retorne APENAS as palavras‑chave, sem nenhum texto adicional.\n"
            "Escreva SEMPRE em inglês."
        )
    )
    try:
        resposta = agente_palavras.run(prompt)
        palavras = resposta.content.strip()
        # Remove pontuação e garante que são no máximo 2 palavras
        import re
        palavras = re.sub(r'[^\w\s]', '', palavras)
        palavras = " ".join(palavras.split()[:2])
        print(f"INFO: Palavras‑chave extraídas: {palavras}")
        return palavras
    except Exception as e:
        print(f"WARN: Erro ao extrair palavras‑chave: {e}")
        return prompt  # fallback seguro


# ---------------------------------------------------------------------------
# Ferramenta PubMed modificada – agora faz 3 consultas
# ---------------------------------------------------------------------------

@tool(name="pesquisa_pubmed")
def ferramenta_pesquisa_pubmed(pergunta: str) -> str:
    """
    Busca artigos acadêmicos no PubMed relacionados à pergunta ou discurso fornecido.
    A ferramenta automaticamente gera e combina os resultados de três pesquisas
    independentes com termos complementares, retornando um resumo dos artigos encontrados.
    Utilize este recurso sempre que precisar embasar uma análise com evidências da literatura.
    """
    print(f"INFO: Ferramenta acionada com pergunta: {pergunta}")

    # Gera três conjuntos de palavras‑chave diferentes
    queries = []
    palavras_usadas = []

    for i in range(3):
        query = extrair_palavras_chave(pergunta, palavras_usadas)
        queries.append(query)
        # Extrai as palavras individuais para evitar repetição
        for palavra in query.split():
            if palavra.lower() not in palavras_usadas:
                palavras_usadas.append(palavra.lower())

    print(f"INFO: Queries geradas: {queries}")

    # Executa as três consultas e combina os resultados
    todos_artigos = []
    for query in queries:
        print(f"INFO: Pesquisando PubMed com: {query}")
        artigos = _fazer_consulta_pubmed(query, max_results=5)
        todos_artigos.extend(artigos)

    if not todos_artigos:
        return "Nenhum artigo encontrado no PubMed para este tópico."

    # Remove duplicatas (baseado no PMID)
    artigos_unicos = []
    pmids_vistos = set()
    for artigo in todos_artigos:
        if artigo["pmid"] not in pmids_vistos:
            pmids_vistos.add(artigo["pmid"])
            artigos_unicos.append(artigo)

    # Formata a resposta
    resposta = []
    for i, paper in enumerate(artigos_unicos, 1):
        resposta.append(f"### {i}. {paper['titulo']}")
        resposta.append(f"**Autores:** {paper['autores']}")
        resposta.append(f"**Publicado em:** {paper['data']}")
        resposta.append(f"**Resumo:** {paper['resumo']}")
        resposta.append(f"**Link:** {paper['link']}")
        resposta.append("")

    resposta_completa = "\n".join(resposta)
    print(f"INFO: Total de artigos únicos recuperados: {len(artigos_unicos)}")
    return resposta_completa


# ---------------------------------------------------------------------------
# Agente principal (instruções mantidas)
# ---------------------------------------------------------------------------

agente_principal = Agent(
    model=Ollama(id="llama3.1", options={"temperature": 0.4}),
    tools=[ferramenta_pesquisa_pubmed],
    instructions=(
        "Você é um sistema especializado em análise crítica de discursos sobre "
        "esteroides anabólicos androgênicos (EAA/AAS) e aminoácidos essenciais (EAA).\n\n"
        "Seu objetivo é identificar desinformação, falácias lógicas e alegações sem "
        "evidência clínica, sempre se baseando em literatura científica.\n\n"
        "**Para cada interação, siga este fluxo:**\n"
        "1. Use a ferramenta 'pesquisa_pubmed' para obter artigos relevantes.\n"
        "2. Analise o discurso do usuário em busca de:\n"
        "   - Falácias (apelo à autoridade, evidência anedótica, generalização indevida)\n"
        "   - Promessas sem base clínica\n"
        "   - Minimização de riscos conhecidos\n"
        "   - Recomendação de uso não terapêutico de esteroides\n"
        "3. Compare as alegações com as evidências dos artigos encontrados.\n"
        "4. Produza uma resposta estruturada contendo:\n"
        "   - Classificação do tipo de discurso (científico, marketing, opinião, etc.)\n"
        "   - Principais falácias identificadas (em formato de lista ou tabela)\n"
        "   - Evidências científicas relevantes (citando os artigos do PubMed com link)\n"
        "   - Conclusão crítica e alerta ético quando cabível\n"
        "   - Referências acadêmicas\n\n"
        "Lembre-se: recomendar esteroides fora de contexto médico, prometer benefícios "
        "sem evidência ou minimizar riscos cardiovasculares/hormonais é automaticamente "
        "considerado problemático. Sempre advirta sobre os perigos com base na literatura.\n"
        "Responda sempre em português, de forma clara e estruturada."
    ),
    markdown=True
)


# ---------------------------------------------------------------------------
# Execução principal
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    print("=" * 60)
    print("Sistema de Análise Crítica de Discursos (PubMed × 3 consultas)")
    print("=" * 60)
    pergunta = input("Digite uma afirmação, discurso ou pergunta sobre EAA/AAS: ")
    resposta = agente_principal.run(pergunta)
    print("\n" + "=" * 60)
    print(resposta.content)

Sistema de Análise Crítica de Discursos (PubMed × 3 consultas)
Digite uma afirmação, discurso ou pergunta sobre EAA/AAS: “Chega de perder tempo na academia! Com o protocolo hormonal certo você multiplica seus ganhos. A testosterona exógena, quando bem administrada, é mais segura que muito remédio que a indústria farmacêutica empurra. O que destrói a saúde é o sedentarismo, não o hormônio. Se você quer resultado de verdade, me chama no direct.”
INFO: Ferramenta acionada com pergunta: testosterona exógena vs remédios farmacêuticos segurança e saúde sedentarismo
INFO: Palavras‑chave extraídas: Anabolicsteroids Cardiovascularrisk
INFO: Palavras‑chave extraídas: Exogenous Testosterone
INFO: Palavras‑chave extraídas: Epidemiologia Clínica
INFO: Queries geradas: ['Anabolicsteroids Cardiovascularrisk', 'Exogenous Testosterone', 'Epidemiologia Clínica']
INFO: Pesquisando PubMed com: Anabolicsteroids Cardiovascularrisk
INFO: Pesquisando PubMed com: Exogenous Testosterone
INFO: Pesquisando PubMed